# Numba lifetimes

Explore **when objects appear, what is shared, and what survives deletion**.

1. Creation: typing context → CPU context → code generator → machine
2. Specializations and per-function options
3. Deleting a dispatcher


## creation

In [60]:
import sys
import numba
import llvmlite
from numba import jit
from numba.core.registry import cpu_target
from llvmlite import binding as llvm

{
    "typing_cached": "_toplevel_typing_context" in vars(cpu_target),
    "CPU_context_cached": "_toplevel_target_context" in vars(cpu_target),
    "Numba": (numba.__version__, numba.__file__),
    "llvmlite": (llvmlite.__version__, llvmlite.__file__),
    "LLVM": llvm.llvm_version_info,
}

{'typing_cached': True,
 'CPU_context_cached': True,
 'Numba': ('0.66.0',
  '/Users/mac357/miniconda3/envs/numba_main/lib/python3.14/site-packages/numba/__init__.py'),
 'llvmlite': ('0.51.0dev0',
  '/Users/mac357/miniconda3/envs/numba_main/lib/python3.14/site-packages/llvmlite/__init__.py'),
 'LLVM': (22, 1, 0)}

Now decorate a function, without calling it. Inspect its contexts, code generator,
and machine. An empty signature list means no specialization has been compiled yet.

Naming trap: `add_one._tm` is a **type manager**. The LLVM target machine is
`add_one.targetctx.codegen()._tm`.

In [61]:
@jit
def add_one(x):
    return x + 1

context = add_one.targetctx
codegen = context.codegen()
machine = codegen._tm
{
    "CPU_context_cached": "_toplevel_target_context" in vars(cpu_target),
    "signatures": add_one.signatures,
    "same_typing_context": add_one.typingctx is typing_context,
    "machine_type": type(machine).__name__,
    "machine_closed": machine.closed,
}

{'CPU_context_cached': True,
 'signatures': [],
 'same_typing_context': True,
 'machine_type': 'TargetMachine',
 'machine_closed': False}

Call it with an integer and a float. Do the signatures change? Does the machine change?

In [62]:
values = [add_one(2), add_one(2.5)]
{
    "results": values,
    "signatures": add_one.signatures,
    "same_machine": add_one.targetctx.codegen()._tm is machine,
}

{'results': [3, 3.5],
 'signatures': [(int64,), (float64,)],
 'same_machine': True}

In [63]:
add_one.overloads

OrderedDict([((int64,),
              CompileResult(typing_context=<numba.core.typing.context.Context object at 0x10c251d30>, target_context=<numba.core.cpu.CPUContext object at 0x10dece190>, entry_point=<built-in method add_one of _dynfunc._Closure object at 0x10ed23c40>, typing_error=None, type_annotation=<numba.core.annotations.type_annotations.TypeAnnotation object at 0x10ed66530>, signature=(int64,) -> int64, objectmode=False, lifted=(), fndesc=<function descriptor 'add_one$25'>, library=<Library 'add_one' at 0x10ecb05d0>, call_helper=None, environment=<Environment '_ZN08NumbaEnv8__main__7add_oneB3v25B38c8tJTIeFIjxB2IKSgI4CrvQClQZ6FczSBAA_3dEx' >, metadata={'parfor_diagnostics': ParforDiagnostics, 'parfors': {}, 'pipeline_times': {'nopython': OrderedDict({'0_translate_bytecode': pass_timings(init=1.416949089616537e-06, run=0.00025158299831673503, finalize=8.33999365568161e-07), '1_fixup_args': pass_timings(init=5.839974619448185e-07, run=1.0830117389559746e-06, finalize=2.50001903

## Whats shared between specializations?

In [64]:
compiled_results = list(add_one.overloads.values())
compilation_contexts = [result.target_context for result in compiled_results]
libraries = [result.library for result in compiled_results]
codegens = [context.codegen() for context in compilation_contexts]
objects = {
    "compilation contexts": compilation_contexts,
    "code libraries": libraries,
    "LLVM modules": [library._final_module for library in libraries],
    "typing contexts": [context.typing_context for context in compilation_contexts],
    "code generators": codegens,
    "engines": [codegen._engine for codegen in codegens],
    "target machines": [codegen._tm for codegen in codegens],
}
{name: len({id(obj) for obj in instances}) for name, instances in objects.items()}

{'compilation contexts': 2,
 'code libraries': 2,
 'LLVM modules': 2,
 'typing contexts': 1,
 'code generators': 1,
 'engines': 1,
 'target machines': 1}

In [65]:
{
    "compilation_is_root": compilation_contexts[0] is context,
    "compilations_same_context": compilation_contexts[0] is compilation_contexts[1],
    "compilation_uses_root_codegen": compilation_contexts[0].codegen() is codegen,
    "dispatcher_uses_root_context": add_one.targetctx is context,
}

{'compilation_is_root': False,
 'compilations_same_context': False,
 'compilation_uses_root_codegen': True,
 'dispatcher_uses_root_context': True}

## 3. Different function options, same machine?

Compile the same Python function with strict and relaxed floating-point options.
This tests an existing per-function policy; it does not establish how a vector-library
choice should be represented.

In [66]:
def add_float(x):
    return x + 1.0

strict = jit(fastmath=False)(add_float)
relaxed = jit(fastmath=True)(add_float)
{"strict_result": strict(2.0), "relaxed_result": relaxed(2.0)}

{'strict_result': 3.0, 'relaxed_result': 3.0}

In [67]:
strict_context = next(iter(strict.overloads.values())).target_context
relaxed_context = next(iter(relaxed.overloads.values())).target_context
{
    "same_context": strict_context is relaxed_context,
    "same_codegen": strict_context.codegen() is relaxed_context.codegen(),
    "same_machine": strict_context.codegen()._tm is relaxed_context.codegen()._tm,
    "strict_fastmath": bool(strict_context.fastmath),
    "relaxed_fastmath": bool(relaxed_context.fastmath),
}

{'same_context': False,
 'same_codegen': True,
 'same_machine': True,
 'strict_fastmath': False,
 'relaxed_fastmath': True}

Inspect the actual LLVM addition instructions. 

In [68]:
for name, function in (("strict", strict), ("relaxed", relaxed)):
    ir = function.inspect_llvm(function.signatures[0])
    additions = [line.strip() for line in ir.splitlines() if "fadd " in line]
    print(name, additions)

strict ['%.5 = fadd double %arg.x, 1.000000e+00']
relaxed ['%.5 = fadd fast double %arg.x, 1.000000e+00']


## 4. What survives deleting a dispatcher?

Use a **new temporary function**, because earlier cells intentionally retain `add_one`'s
compilation results. A small setup function keeps temporary strong references local.
Only the dispatcher and weak references leave that setup function.

Do not display the dispatcher or its compilation objects before deleting them:
Jupyter's `Out`/`_` history can retain them. The outputs here contain booleans only.

In [69]:
import gc
import weakref


def prepare_deletion():
    @jit
    def temporary(x):
        return x + 1

    temporary(2)
    result = next(iter(temporary.overloads.values()))
    references = {
        "dispatcher": weakref.ref(temporary),
        "compilation context": weakref.ref(result.target_context),
        "code library": weakref.ref(result.library),
        "LLVM module": weakref.ref(result.library._final_module),
        "root context": weakref.ref(context),
        "code generator": weakref.ref(codegen),
        "engine": weakref.ref(codegen._engine),
        "target machine": weakref.ref(machine),
    }
    return temporary, references

In [70]:
temporary, weak_objects = prepare_deletion()
{name: reference() is not None for name, reference in weak_objects.items()}

{'dispatcher': True,
 'compilation context': True,
 'code library': True,
 'LLVM module': True,
 'root context': True,
 'code generator': True,
 'engine': True,
 'target machine': True}

Delete the strong reference, collect Python garbage, then inspect the weak references.
To repeat this deletion cell, rerun the preceding setup call first.

In [71]:
del temporary
gc.collect()
{name: reference() is not None for name, reference in weak_objects.items()}

{'dispatcher': False,
 'compilation context': False,
 'code library': False,
 'LLVM module': True,
 'root context': True,
 'code generator': True,
 'engine': True,
 'target machine': True}

In [72]:
with codegen._pass_builder() as first, codegen._pass_builder() as second:
    print("same builder:", first is second)
    print("same machine:", first._tm is second._tm is machine)

{
    "builders_closed": (first.closed, second.closed),
    "machine_closed": machine.closed,
}

same builder: False
same machine: True


{'builders_closed': (True, True), 'machine_closed': False}

In [73]:
@jit
def square(x):
    return x * x

{
    "result": square(4),
    "same_machine_after_compilation": square.targetctx.codegen()._tm is machine,
}

{'result': 16, 'same_machine_after_compilation': True}